# Librerías

In [18]:
using CSV
using DataFrames
using LightGBM
using BayesianOptimization

# Seteo experimento

In [55]:
PARAM = Dict();
PARAM["experimento"] = "seg-001";
PARAM["semilla_primigenia"] = 102191;

In [ ]:
#setwd("/content/buckets/b1/exp")
#experimento_folder <- PARAM$experimento
#dir.create(experimento_folder, showWarnings=FALSE)
#setwd( paste0("/content/buckets/b1/exp/", experimento_folder ))

# Lectura de archivo

Por ahora el dataset viene con la clase ternaria (después ver si agregar su cálculo)

In [64]:
dataset = DataFrame(CSV.File("buckets/b1/datasets/analistasr_competencia_2026.csv.gz"));

In [65]:
# Paso 1: Agrupar y sumar los valores
df_agrupado = combine(groupby(dataset,[:clase_ternaria, :foto_mes]), nrow => :len)

# Paso 2: Hacer el "unstack" (Pivot)
# Parámetros: (dataframe, fila, columna_a_expandir, valores_a_rellenar)
unstack(df_agrupado, :foto_mes, :clase_ternaria, :len,allowmissing=true)

Row,foto_mes,CONTINUA,BAJA+1,BAJA+2,missing
,Int64,Int64?,Int64?,Int64?,Int64?
1,201901,122682,654,695,missing
2,201902,123735,697,692,missing
3,201903,124265,696,771,missing
4,201904,124897,773,576,missing
5,201905,125634,577,665,missing
6,201906,127219,672,583,missing
7,201907,128708,588,661,missing
8,201908,130686,664,537,missing
9,201909,132387,542,609,missing


# Feature engineering

por ahora queda vacío

# modelado

## target engineering

In [86]:
# paso la clase a binaria que tome valores {0,1}  enteros
#  BAJA+1 y BAJA+2  son  1,   CONTINUA es 0
#  a partir de ahora ya NO puedo cortar  por prob(BAJA+2) > 1/40

transform!(dataset, :clase_ternaria => ByRow(passmissing(x -> x == "CONTINUA" ? 0 : 1)) => :clase_01);

## training strategy

In [88]:
PARAM["training_strategy"] = Dict();
PARAM["training_strategy"]["testing"] = [202107];

PARAM["training_strategy"]["training"] = [
  201901, 201902, 201903, 201904, 201905, 201906,
  201907, 201908, 201909, 201910, 201911, 201912,
  202001, 202002, 202003, 202004, 202005, 202006,
  202007, 202008, 202009, 202010, 202011, 202012,
  202101, 202102, 202103, 202104, 202105
]

PARAM["training_strategy"]["undersampling"] = 0.05;

In [80]:
# los campos en los que se entrena
campos_buenos  = copy(setdiff(names(dataset), ["clase_ternaria", "clase01", ]));

In [ ]:
# Undersampling, van todos los "BAJA+1" y "BAJA+2" y solo algunos "CONTINIA"

set.seed(PARAM$semilla_primigenia, kind = "L'Ecuyer-CMRG")
dataset[, azar := runif(nrow(dataset))]
dataset[, training := 0L]

dataset[  foto_mes %in%  PARAM$trainingstrategy$training &
  (azar <= PARAM$trainingstrategy$undersampling | clase_ternaria %in% c("BAJA+1", "BAJA+2")),
  training := 1L
]
